# 10v5_01 — Scale feature build (driver for `scripts/v5_build_scale_features.py`)

This notebook is a thin, resumable wrapper around the offline script. The script is the source
of truth and is the recommended way to run long builds (a terminal survives notebook restarts):

```bash
PYTHONPATH=src PYTHONNOUSERSITE=1 python scripts/v5_build_scale_features.py --manifest TL_EVAL
PYTHONPATH=src PYTHONNOUSERSITE=1 python scripts/v5_build_scale_features.py --manifest TL_1K
PYTHONPATH=src PYTHONNOUSERSITE=1 python scripts/v5_build_scale_features.py --manifest RND_1K
PYTHONPATH=src PYTHONNOUSERSITE=1 python scripts/v5_build_scale_features.py --manifest TL_3K
PYTHONPATH=src PYTHONNOUSERSITE=1 python scripts/v5_build_scale_features.py --manifest TL_10K
PYTHONPATH=src PYTHONNOUSERSITE=1 python scripts/v5_build_scale_features.py --manifest MOL_DEV
```

The evidence machinery is the v4b one, unchanged. Every finished 50-query chunk is a
fingerprinted shard that is skipped on re-run. Nested manifests only build their delta.
**Do not edit `src/casmi` between builds**, because the code version is part of every shard
fingerprint.

In [1]:
import sys, json, importlib.util
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'src').exists() and (REPO_ROOT.parent / 'src').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
import pandas as pd
from casmi.paths import get_project_paths
from casmi.qcr.context import v5_paths

D = v5_paths(get_project_paths())
spec = importlib.util.spec_from_file_location('v5_build', REPO_ROOT / 'scripts' / 'v5_build_scale_features.py')
v5_build = importlib.util.module_from_spec(spec)
spec.loader.exec_module(v5_build)

# edit this list as the science track progresses (TL_1K first proves the whole pipeline end to end)
BUILD = ['TL_EVAL', 'TL_1K', 'RND_1K']
# BUILD = ['TL_3K']
# BUILD = ['TL_10K']
# BUILD = ['MOL_DEV']

In [2]:
results = {}
for name in BUILD:
    print('=' * 80, '\n', name)
    results[name] = v5_build.main(['--manifest', name])

 TL_EVAL
[v5] manifest=TL_EVAL units=['TL_EVAL'] evidence=14cc3ca0bf9be807b84c9e1e ppm=50
[v5] unit TL_EVAL: {'unit': 'TL_EVAL', 'manifest': 'TL_EVAL', 'n_queries': 1000, 'n_queries_with_candidates': 1000, 'n_pool_rows': 359875, 'pool_fingerprint': 'cbec2e0a8e06e7c292fb5d02', 'qcr_fingerprint': 'c5fb4f8b0f34a504', 'n_chunks': 20, 'n_chunks_built': 20, 'n_chunks_resumed': 0, 'max_chunk_rss_gb': 2.0717506408691406, 'wall_seconds': 1905.7669956684113, 'finished_at': '2026-09-24T23:03:45.319141+00:00'}
[v5] TL_EVAL: 1,000 queries (1,000 with candidates), 359,875 rows -> C:\Users\myben\OneDrive\Documents\Enveda\outputs\v5\features\TL_EVAL_mirror_aware.parquet  [31.8 min]
 TL_1K
[v5] manifest=TL_1K units=['TL_1K'] evidence=14cc3ca0bf9be807b84c9e1e ppm=50
[v5] unit TL_1K: {'unit': 'TL_1K', 'manifest': 'TL_1K', 'n_queries': 1000, 'n_queries_with_candidates': 1000, 'n_pool_rows': 425332, 'pool_fingerprint': '2b74607202008c81922949c4', 'qcr_fingerprint': 'ac8376d97c2e937d', 'n_chunks': 20, 'n_ch

## Build status (every manifest, from the persisted build records)

In [3]:
rows = []
for name in ('TL_EVAL', 'TL_1K', 'TL_3K', 'TL_10K', 'RND_1K', 'MOL_DEV'):
    p = D['features'] / f'{name}_mirror_aware.build.json'
    if p.exists():
        r = json.loads(p.read_text(encoding='utf-8'))
        rows.append({'manifest': name, 'status': 'BUILT', 'n_queries': r['n_queries'], 'with_candidates': r['n_queries_with_candidates'],
                     'rows': r['n_rows'], 'minutes': r['wall_seconds'] / 60,
                     'max_chunk_rss_gb': max((u.get('max_chunk_rss_gb') or 0) for u in r['units']),
                     'chunks_built': sum(u['n_chunks_built'] for u in r['units']), 'chunks_resumed': sum(u['n_chunks_resumed'] for u in r['units'])})
    else:
        rows.append({'manifest': name, 'status': 'NOT_BUILT'})
display(pd.DataFrame(rows))

,manifest,status,n_queries,with_candidates,rows,minutes,max_chunk_rss_gb,chunks_built,chunks_resumed
0,TL_EVAL,BUILT,1000.0,1000.0,359875.0,31.818449,2.071751,20.0,0.0
1,TL_1K,BUILT,1000.0,1000.0,425332.0,37.421309,2.236935,20.0,0.0
2,TL_3K,NOT_BUILT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,TL_10K,NOT_BUILT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,RND_1K,BUILT,1000.0,1000.0,241964.0,20.388276,2.015106,20.0,0.0
5,MOL_DEV,NOT_BUILT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
